# GoldRegime X — Interactive Research Notebook

> Interactive exploration sandbox for XAUUSD hybrid HMM + XGBoost trading system.
> Run **Section 1** first to load data and models, then explore any section below.

| Section | Feature | Speed |
|---------|---------|-------|
| 1 | Data & Model Loader | ~5 s |
| 2 | Equity Curve Explorer | instant |
| 3 | WFO Window Analysis | ~2–3 min |
| 4 | Feature & Regime Explorer | instant |
| 5 | Parameter Sensitivity | ~30–60 s |

In [12]:
import sys, os, warnings
_here = os.path.abspath('.')
if os.path.basename(_here).lower() == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

from src.processor   import load_data_with_hmm_labels
from src.optimizer   import (get_best_params, _run_wfo, WFO_PARAMS,
                              WFO_PARAMS_FAST, CV_FOLDS, compute_cpcv_score)
from src.engine_hmm  import (load_model, predict_states, fit_hmm,
                               get_model_path, STATE_NAMES_2, STATE_NAMES_3, STATE_NAMES_4)
from src.engine_xgb  import (load_xgb_ensemble, train_xgb_ensemble, prepare_features,
                               get_predictions_ensemble, get_ensemble_path)
from src.backtester  import vectorized_backtest

_CACHE = {}
STATE_NAMES_MAP = {2: STATE_NAMES_2, 3: STATE_NAMES_3, 4: STATE_NAMES_4}
STATE_COLORS    = {0: '#27AE60', 1: '#E74C3C', 2: '#7F8C8D', 3: '#F39C12'}
print('Setup complete — working directory:', os.getcwd())

Setup complete — working directory: c:\GoldRegime_X


---
## Section 1 — Data & Model Loader
Select timeframe, broker, and balance then click **Load Data & Models**.
Processed parquet + saved HMM/XGB models are loaded; if missing they are fitted from scratch.

In [13]:
_style = {'description_width': 'initial'}
_layout = widgets.Layout(width='220px')

tf_w       = widgets.Dropdown(options=['H1', 'M15', 'M5'], value='H1',
                               description='Timeframe:', style=_style, layout=_layout)
broker_w   = widgets.Dropdown(options=['headway_cent', 'standard'], value='headway_cent',
                               description='Broker:', style=_style, layout=_layout)
wfo_mode_w = widgets.Dropdown(options=['standard', 'fast'], value='standard',
                               description='WFO Mode:', style=_style, layout=_layout)
balance_w = widgets.IntSlider(min=10, max=500, step=5, value=15,
                               description='Balance $:', style=_style,
                               layout=widgets.Layout(width='380px'), continuous_update=False)
load_btn  = widgets.Button(description='Load Data & Models', button_style='primary',
                            layout=widgets.Layout(width='200px'))
out1      = widgets.Output()


def _do_load(_):
    with out1:
        clear_output(wait=True)
        tf, broker, balance = tf_w.value, broker_w.value, balance_w.value
        wfo_mode = wfo_mode_w.value
        print(f'Loading {tf} / {broker} / ${balance} ...')

        try:
            df = load_data_with_hmm_labels(tf, broker)
        except FileNotFoundError as exc:
            print(f'ERROR: {exc}'); return

        try:
            params = get_best_params(balance, broker, tf)
            print(f'Params: n_states={params.get("n_states",3)}  '
                  f'obs_cov={params.get("obs_cov",1.0):.4f}  '
                  f'lr={params.get("learning_rate",0.1):.4f}')
        except Exception:
            params = {}
            print('No Optuna study found — using built-in defaults')

        try:
            hmm_model = load_model(get_model_path(tf, broker))
        except Exception:
            n_st = int(params.get('n_states', 3))
            hmm_model, _, _ = fit_hmm(df, n_states=n_st, tf=tf)
            print('No saved HMM — fitted from scratch')

        try:
            xgb_models, xgb_thresholds, xgb_metrics = load_xgb_ensemble(get_ensemble_path(tf, broker))
        except Exception:
            states_tmp = predict_states(hmm_model, df)
            X_tmp, y_tmp, _, _ = prepare_features(df, states_tmp, tf=tf)
            kws = {k: params[k] for k in ['max_depth','learning_rate','n_estimators',
                   'subsample','min_child_weight','gamma','reg_alpha',
                   'colsample_bytree','scale_pos_weight'] if k in params}
            xgb_models, xgb_thresholds, xgb_metrics = train_xgb_ensemble(X_tmp, y_tmp, **kws)
            print('No saved XGB — trained from scratch')

        states = (df['hmm_state'].values if 'hmm_state' in df.columns
                  else predict_states(hmm_model, df))
        X, y, df_al, scaler = prepare_features(df, states, tf=tf)
        _, probs = get_predictions_ensemble(xgb_models, xgb_thresholds, X)
        states_al = pd.Series(states, index=df.index).loc[df_al.index].values
        split_idx = xgb_metrics.get('split_idx', int(len(X) * 0.8))

        _CACHE.update(dict(
            df=df, df_aligned=df_al, states=states, states_aligned=states_al,
            probs=probs, split_idx=split_idx, params=params, tf=tf,
            broker=broker, balance=balance, hmm_model=hmm_model,
            xgb_models=xgb_models, xgb_thresholds=xgb_thresholds, xgb_metrics=xgb_metrics, scaler=scaler, X=X, y=y,
            wfo_mode=wfo_mode,
        ))

        n_st  = int(states.max()) + 1
        sname = STATE_NAMES_MAP.get(n_st, STATE_NAMES_3)
        cnts  = pd.Series(states).value_counts().sort_index()
        pct   = (cnts / len(states) * 100).round(1)
        print(f'\n{"="*56}')
        print(f'Rows:     {len(df):>10,}   ({df.index[0].date()} → {df.index[-1].date()})')
        print(f'Features: {list(X.columns)}')
        print(f'IS split: {split_idx:>10,}   ({split_idx/len(X)*100:.0f}%)')
        print(f'Regimes:  ' + '  |  '.join(f'{sname[s]}={pct[s]:.1f}%' for s in cnts.index))
        print(f'{"="*56}')
        print('Done — run any section below.')


load_btn.on_click(_do_load)
display(widgets.VBox([
    widgets.HBox([tf_w, broker_w, wfo_mode_w]),
    balance_w,
    load_btn, out1,
]))

---
## Section 2 — Equity Curve Explorer
Adjust **Balance** or **Broker** — the equity curve re-runs automatically (backtest only, no retraining).
Orange dashed line = IS / OOS split.

In [14]:
bal2_w   = widgets.IntSlider(min=10, max=500, step=5, value=15, description='Balance $:',
                              style={'description_width': 'initial'},
                              layout=widgets.Layout(width='380px'), continuous_update=False)
broker2_w = widgets.Dropdown(options=['headway_cent', 'standard'], value='headway_cent',
                              description='Broker:', style={'description_width': 'initial'},
                              layout=widgets.Layout(width='220px'))


def _plot_equity(balance, broker):
    if 'df_aligned' not in _CACHE:
        print('Run Section 1 first.'); return

    df_al     = _CACHE['df_aligned']
    probs     = _CACHE['probs']
    states_al = _CACHE['states_aligned']
    split_idx = _CACHE['split_idx']
    tf        = _CACHE['tf']

    r  = vectorized_backtest(df_al, probs, states_al, split_idx=split_idx,
                              account_size=balance, broker=broker, tf=tf)
    ts = pd.DatetimeIndex(r['equity_timestamps'])
    eq = r['equity_values']
    bh = balance * np.exp(np.cumsum(df_al['log_return'].fillna(0).values))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})
    if split_idx < len(ts):
        for ax in (ax1, ax2):
            ax.axvline(ts[split_idx], color='orange', ls='--', lw=1.5)
    ax1.plot(ts, eq, lw=1.5, label='Strategy', color='#3498DB')
    ax1.plot(ts, bh, lw=1.0, ls=':', label='Buy & Hold', color='#E67E22', alpha=0.7)
    ax1.axhline(balance, color='grey', lw=0.6, alpha=0.5)
    ax1.set_ylabel('Equity ($)')
    ax1.legend(loc='upper left', fontsize=9)

    peak = np.maximum.accumulate(eq)
    dd   = (eq - peak) / peak * 100
    ax2.fill_between(ts, dd, 0, color='#E74C3C', alpha=0.4)
    ax2.set_ylabel('Drawdown (%)')
    ax2.set_xlabel('Date')

    is_sh  = r.get('is_sharpe_ratio',  r['sharpe_ratio'])
    oos_sh = r.get('oos_sharpe_ratio', r['sharpe_ratio'])
    is_n   = r.get('is_n_trades',  r['n_trades'])
    oos_n  = r.get('oos_n_trades', r['n_trades'])
    fig.suptitle(
        f'{tf} | {broker} | ${balance}   '
        f'IS Sharpe: {is_sh:.2f} ({is_n} trades)  |  '
        f'OOS Sharpe: {oos_sh:.2f} ({oos_n} trades)  |  '
        f'Max DD: {r["max_drawdown"]*100:.1f}%  |  WR: {r["win_rate"]*100:.1f}%',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

    is_dd  = r.get('is_max_drawdown',  r['max_drawdown'])
    oos_dd = r.get('oos_max_drawdown', r['max_drawdown'])
    is_wr  = r.get('is_win_rate',  r['win_rate'])
    oos_wr = r.get('oos_win_rate', r['win_rate'])
    rows = [
        ('Sharpe',          f'{is_sh:.3f}',         f'{oos_sh:.3f}',       f'{r["sharpe_ratio"]:.3f}'),
        ('Max Drawdown',    f'{is_dd*100:.1f}%',    f'{oos_dd*100:.1f}%',  f'{r["max_drawdown"]*100:.1f}%'),
        ('Win Rate',        f'{is_wr*100:.1f}%',    f'{oos_wr*100:.1f}%',  f'{r["win_rate"]*100:.1f}%'),
        ('Trades',          str(is_n),               str(oos_n),            str(r['n_trades'])),
        ('Profit Factor',   '–',                     '–',                   f'{r.get("profit_factor", 0):.2f}'),
        ('Recovery Factor', '–',                     '–',                   f'{r.get("recovery_factor", 0):.2f}'),
        ('Avg Efficiency',  '–',                     '–',                   f'{r.get("avg_efficiency", 0):.2f}x'),
    ]
    print(f'  {"Metric":<20} {"IS":>10} {"OOS":>10} {"Total":>10}')
    print('  ' + '─' * 54)
    for m, i, o, t in rows:
        print(f'  {m:<20} {i:>10} {o:>10} {t:>10}')


out2 = widgets.interactive_output(_plot_equity, {'balance': bal2_w, 'broker': broker2_w})
display(widgets.VBox([widgets.HBox([bal2_w, broker2_w]), out2]))

---
## Section 3 — WFO Window Analysis
Click **Run WFO Analysis** to evaluate the best Optuna params across all rolling windows (~2–3 min).
Then use the **Window #** slider to drill into any OOS window's equity curve.

In [15]:
wfo_btn    = widgets.Button(description='Run WFO Analysis', button_style='warning',
                             layout=widgets.Layout(width='180px'))
out3_wfo   = widgets.Output()
win_slider = widgets.IntSlider(min=0, max=25, step=1, value=0, description='Window #:',
                                style={'description_width': 'initial'},
                                layout=widgets.Layout(width='380px'), continuous_update=False)
out3_drill = widgets.Output()


def _run_wfo_analysis(_):
    with out3_wfo:
        clear_output(wait=True)
        if 'df_aligned' not in _CACHE:
            print('Run Section 1 first.'); return

        df_al   = _CACHE['df_aligned']
        params  = _CACHE['params']
        tf      = _CACHE['tf']
        broker  = _CACHE['broker']
        balance = _CACHE['balance']

        if not params:
            print('No Optuna params — run optimize first.'); return

        wfo_mode = _CACHE.get('wfo_mode', 'standard')
        cfg_map  = WFO_PARAMS_FAST if wfo_mode == 'fast' else WFO_PARAMS
        cfg      = cfg_map.get(tf.upper(), WFO_PARAMS['H1'])
        n_states = int(params.get('n_states', 3))
        xgb_kws  = {k: params[k] for k in ['max_depth','learning_rate','n_estimators',
                    'subsample','min_child_weight','gamma','reg_alpha',
                    'colsample_bytree','scale_pos_weight'] if k in params}

        print(f'WFO [{wfo_mode}]  tf={tf}  IS={cfg["is_bars"]}bars  OOS={cfg["oos_bars"]}bars  n_states={n_states}')
        print('Running ... (each window fits its own HMM — takes a few minutes)')

        wfo = _run_wfo(df_al, n_states=n_states, tf=tf, balance=balance, broker=broker,
                       xgb_kwargs=xgb_kws, **cfg)
        _CACHE['wfo_result'] = wfo
        _CACHE['wfo_cfg']    = cfg

        # Update window slider range
        win_slider.max = max(0, len(wfo['window_scores']) - 1)

        scores = wfo['window_scores']
        colors = ['#27AE60' if s > 0 else '#E74C3C' for s in scores]

        # Build OOS start-year labels
        is_b, oos_b, emb_b, step_b = (cfg[k] for k in
                                       ['is_bars','oos_bars','embargo_bars','step_bars'])
        labels = []
        for i in range(len(scores)):
            oos_st = i * step_b + is_b + emb_b
            if oos_st < len(df_al):
                labels.append(str(df_al.index[oos_st].date())[:7])
            else:
                labels.append('')

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(range(len(scores)), scores, color=colors, edgecolor='white', lw=0.5)
        ax.axhline(0, color='black', lw=0.8)
        ax.axhline(wfo['wfo_score'], color='orange', ls='--', lw=1.5,
                   label=f'WFO score = {wfo["wfo_score"]:.3f}')
        ax.set_xticks(range(len(scores)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        ax.set_xlabel('OOS window start')
        ax.set_ylabel('OOS Complex Criterion')
        ax.set_title(
            f'WFO Window Scores — {tf}  |  '
            f'valid: {wfo["n_valid_windows"]}/{wfo["n_windows"]}  |  '
            f'std_sharpe={wfo["std_sharpe"]:.2f}  |  '
            f'WFE={wfo["wfe_ratio"]:.2f}'
        )
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(f'WFO score:     {wfo["wfo_score"]:.4f}  (median – 0.20×std)')
        print(f'WFE ratio:     {wfo["wfe_ratio"]:.4f}  (>0.5 = robust)')
        print(f'Std Sharpe:    {wfo["std_sharpe"]:.4f}  (lower = consistent)')
        print(f'Median trades: {wfo["median_trades"]}')
        print(f'Valid windows: {wfo["n_valid_windows"]}/{wfo["n_windows"]}')

        wfe = wfo['wfe_ratio']
        if wfe >= 0.8:
            wfe_label = '✅ Excellent (OOS ≥ 80% of IS CV performance)'
        elif wfe >= 0.5:
            wfe_label = '🟡 Acceptable (OOS ≥ 50% of IS CV performance)'
        elif wfe >= 0.2:
            wfe_label = '🟠 Marginal — consider broader regularization'
        else:
            wfe_label = '🔴 Poor — strong overfitting signal'
        print(f'WFE interpretation: {wfe_label}')


def _drill_window(window_idx):
    with out3_drill:
        clear_output(wait=True)
        if 'wfo_cfg' not in _CACHE:
            print('Run WFO Analysis first.'); return

        df_al   = _CACHE['df_aligned']
        probs   = _CACHE['probs']
        st_al   = _CACHE['states_aligned']
        balance = _CACHE['balance']
        broker  = _CACHE['broker']
        tf      = _CACHE['tf']
        cfg     = _CACHE['wfo_cfg']
        scores  = _CACHE['wfo_result']['window_scores']

        is_b, oos_b, emb_b, step_b = (cfg[k] for k in
                                       ['is_bars','oos_bars','embargo_bars','step_bars'])
        oos_st  = window_idx * step_b + is_b + emb_b
        oos_end = min(oos_st + oos_b, len(df_al))

        if oos_st >= len(df_al):
            print(f'Window {window_idx} is out of data range.'); return

        df_w  = df_al.iloc[oos_st:oos_end]
        pr_w  = probs[oos_st:oos_end]
        st_w  = st_al[oos_st:oos_end]

        r  = vectorized_backtest(df_w, pr_w, st_w, account_size=balance,
                                  broker=broker, tf=tf)
        ts = pd.DatetimeIndex(r['equity_timestamps'])
        eq = r['equity_values']
        bh = balance * np.exp(np.cumsum(df_w['log_return'].fillna(0).values))

        wfo_s = scores[window_idx] if window_idx < len(scores) else float('nan')
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(ts, eq, lw=1.5, label='Strategy', color='#3498DB')
        ax.plot(ts, bh, lw=1.0, ls=':', label='Buy & Hold', color='#E67E22', alpha=0.7)
        ax.axhline(balance, color='grey', lw=0.6, alpha=0.5)
        ax.set_title(
            f'Window {window_idx} OOS: {ts[0].date()} → {ts[-1].date()}  |  '
            f'Sharpe={r["sharpe_ratio"]:.2f}  DD={r["max_drawdown"]*100:.1f}%  '
            f'Trades={r["n_trades"]}  WFO score={wfo_s:.3f}'
        )
        ax.legend(fontsize=9)
        ax.set_ylabel('Equity ($)')
        plt.tight_layout()
        plt.show()


out3_drill_ui = widgets.interactive_output(_drill_window, {'window_idx': win_slider})

wfo_btn.on_click(_run_wfo_analysis)
display(widgets.VBox([
    wfo_btn,
    out3_wfo,
    widgets.Label('Window drill-down (uses globally loaded model, not per-window refitted):'),
    win_slider,
    out3_drill_ui,
]))

---
## Section 4 — Feature & Regime Explorer
Shows regime overlay on price, GMM vol cluster distribution, feature distribution by regime,
and XGB feature importance. Re-execute this cell after loading new data in Section 1.

In [16]:
def _shade_regimes(ax, ts, states):
    boundaries = np.concatenate([[0], np.where(np.diff(states))[0] + 1, [len(states)]])
    for k in range(len(boundaries) - 1):
        i, j = boundaries[k], boundaries[k+1] - 1
        s = int(states[i])
        ax.axvspan(ts[i], ts[j], alpha=0.15, color=STATE_COLORS.get(s, '#888888'), lw=0)


def _plot_regime_explorer():
    if 'df_aligned' not in _CACHE:
        print('Run Section 1 first.'); return

    df_al  = _CACHE['df_aligned']
    states = _CACHE['states_aligned']
    X      = _CACHE['X']
    xgb_m  = _CACHE['xgb_metrics']
    tf     = _CACHE['tf']
    n_st   = int(states.max()) + 1
    snames = STATE_NAMES_MAP.get(n_st, STATE_NAMES_3)
    ts     = df_al.index

    fig = plt.figure(figsize=(16, 13))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.35)

    # ── Price + Regime overlay (full width) ──────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    if 'close' in df_al.columns:
        price = df_al['close'].values
    elif 'Close' in df_al.columns:
        price = df_al['Close'].values
    else:
        price = 100 * np.exp(np.cumsum(df_al['log_return'].fillna(0).values))
    _shade_regimes(ax1, ts, states)
    ax1.plot(ts, price, lw=0.6, color='black', zorder=3)
    patches = [mpatches.Patch(color=STATE_COLORS[s], alpha=0.5, label=n)
               for s, n in snames.items()]
    ax1.legend(handles=patches, loc='upper left', fontsize=9)
    ax1.set_title(f'{tf} Price + HMM Regime Overlay', fontsize=11)
    ax1.set_ylabel('Price')

    # ── Regime distribution pie ───────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    cnts = [int((states == s).sum()) for s in snames]
    pie_cols = [STATE_COLORS[s] for s in snames]
    ax2.pie(cnts, labels=list(snames.values()), colors=pie_cols,
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('Regime Distribution')

    # ── GMM vol cluster distribution ─────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    if 'gmm_vol_cluster' in df_al.columns:
        clusters  = df_al['gmm_vol_cluster'].dropna().astype(int)
        unique_c  = sorted(clusters.unique())
        counts_c  = [int((clusters == c).sum()) for c in unique_c]
        bar_cols  = ['#3498DB', '#E67E22', '#9B59B6', '#1ABC9C'][:len(unique_c)]
        ax3.bar(unique_c, counts_c, color=bar_cols, edgecolor='white')
        for c, cnt in zip(unique_c, counts_c):
            ax3.text(c, cnt + counts_c[0] * 0.01,
                     f'{cnt/len(clusters)*100:.1f}%', ha='center', fontsize=8)
        ax3.set_title('GMM Volatility Cluster Distribution')
        ax3.set_xlabel('Cluster ID')
        ax3.set_ylabel('Bar count')
    else:
        ax3.text(0.5, 0.5, 'gmm_vol_cluster not in data',
                 transform=ax3.transAxes, ha='center', va='center')

    # ── Feature distributions per regime ─────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    feat = 'rsi_slope' if 'rsi_slope' in X.columns else X.columns[1]
    for s, sname in snames.items():
        mask = states == s        # numpy bool — use positional indexing, not .loc
        vals = X[mask][feat].dropna() if mask.sum() > 0 else pd.Series(dtype=float)
        if len(vals) > 0:
            ax4.hist(vals, bins=60, alpha=0.5, label=sname,
                     color=STATE_COLORS[s], density=True)
    ax4.set_title(f'{feat} Distribution per Regime')
    ax4.set_xlabel(feat)
    ax4.set_ylabel('Density')
    ax4.legend(fontsize=9)

    # ── XGB Feature Importance ────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    imp = xgb_m.get('feature_importance', {})
    if imp:
        sorted_imp = sorted(imp.items(), key=lambda x: x[1])
        names_i, vals_i = zip(*sorted_imp)
        colors_i = ['#3498DB' if v > 0.05 else '#BDC3C7' for v in vals_i]
        ax5.barh(names_i, vals_i, color=colors_i)
        ax5.set_title('XGB Feature Importance')
        ax5.set_xlabel('Importance')
    else:
        ax5.text(0.5, 0.5, 'No importance data', transform=ax5.transAxes,
                 ha='center', va='center')

    fig.suptitle(f'Feature & Regime Explorer — {tf}', fontsize=13, y=1.01)
    plt.show()


_plot_regime_explorer()


Run Section 1 first.


---
## Section 5 — Parameter Sensitivity
Adjust sliders then click **Run Comparison** to re-fit HMM + XGB with new params
and compare metrics against the Section 1 baseline (~30–60 s).

In [17]:
_s = {'description_width': 'initial'}
_w = widgets.Layout(width='380px')

n_states5_w       = widgets.IntSlider(min=2, max=4, step=1, value=3,
                                       description='n_states:', style=_s, layout=_w)
depth5_w          = widgets.IntSlider(min=2, max=7, step=1, value=4,
                                       description='max_depth:', style=_s, layout=_w)
lr5_w             = widgets.FloatLogSlider(min=-3.0, max=-0.5, step=0.1, value=0.1, base=10,
                                            description='learning_rate:', style=_s, layout=_w,
                                            readout_format='.4f')
alpha5_w          = widgets.FloatLogSlider(min=-6.0, max=-0.5, step=0.2, value=0.1, base=10,
                                            description='reg_alpha:', style=_s, layout=_w,
                                            readout_format='.5f')
reg_lambda5_w     = widgets.FloatLogSlider(min=-1.0, max=1.5, step=0.1, value=1.0, base=10,
                                            description='reg_lambda:', style=_s, layout=_w,
                                            readout_format='.4f')
min_child_weight5_w = widgets.IntSlider(min=1, max=100, step=1, value=5,
                                         description='min_child_weight:', style=_s, layout=_w)
run_btn5          = widgets.Button(description='Run Comparison', button_style='info',
                                    layout=widgets.Layout(width='160px'))
run_wfo5_btn      = widgets.Button(description='Run WFO Score Comparison', button_style='warning',
                                    layout=widgets.Layout(width='220px'))
out5              = widgets.Output()
out5_wfo          = widgets.Output()


def _run_sensitivity(_):
    with out5:
        clear_output(wait=True)
        if 'df_aligned' not in _CACHE:
            print('Run Section 1 first.'); return

        df_al  = _CACHE['df_aligned']
        df_raw = _CACHE['df']
        params = _CACHE['params']
        tf     = _CACHE['tf']
        broker = _CACHE['broker']
        bal    = _CACHE['balance']

        # Baseline
        print('Baseline backtest ...')
        bl_r = vectorized_backtest(_CACHE['df_aligned'], _CACHE['probs'],
                                    _CACHE['states_aligned'], split_idx=_CACHE['split_idx'],
                                    account_size=bal, broker=broker, tf=tf)

        # Modified: re-fit HMM with new n_states
        n_st_new = n_states5_w.value
        print(f'Fitting HMM n_states={n_st_new} ...')
        new_hmm, new_st_full, _ = fit_hmm(df_raw, n_states=n_st_new, tf=tf)
        new_states = pd.Series(new_st_full, index=df_raw.index).loc[df_al.index].values

        # Modified: re-train XGB with new params
        kws = {k: params.get(k, v) for k, v in [
            ('n_estimators', 200), ('subsample', 0.8),
            ('min_child_weight', 5), ('colsample_bytree', 0.8), ('scale_pos_weight', 1.0),
        ]}
        kws['max_depth']        = depth5_w.value
        kws['learning_rate']    = lr5_w.value
        kws['gamma']            = params.get('gamma', 0.1)
        kws['reg_alpha']        = alpha5_w.value
        kws['reg_lambda']       = reg_lambda5_w.value
        kws['min_child_weight'] = min_child_weight5_w.value

        print(f'Training XGB depth={depth5_w.value} lr={lr5_w.value:.4f} '
              f'alpha={alpha5_w.value:.5f} ...')
        X_new, y_new, df_al2, _ = prepare_features(df_al, new_states, tf=tf)
        new_models, new_thresholds, new_met = train_xgb_ensemble(X_new, y_new, **kws)
        _, new_probs = get_predictions_ensemble(new_models, new_thresholds, X_new)
        new_st2  = pd.Series(new_states, index=df_al.index).loc[df_al2.index].values
        new_spl  = new_met.get('split_idx', int(len(X_new) * 0.8))

        print('Backtesting modified config ...')
        new_r = vectorized_backtest(df_al2, new_probs, new_st2,
                                     split_idx=new_spl, account_size=bal,
                                     broker=broker, tf=tf)

        # Comparison bar chart
        bl_oos_sh  = bl_r.get('oos_sharpe_ratio',  bl_r['sharpe_ratio'])
        new_oos_sh = new_r.get('oos_sharpe_ratio', new_r['sharpe_ratio'])
        metrics_cmp = {
            'OOS Sharpe':    [bl_oos_sh,                         new_oos_sh],
            'Max DD %':      [bl_r['max_drawdown'] * 100,        new_r['max_drawdown'] * 100],
            'Win Rate %':    [bl_r['win_rate'] * 100,            new_r['win_rate'] * 100],
            'Profit Factor': [bl_r.get('profit_factor', 0),      new_r.get('profit_factor', 0)],
            'Recovery F.':   [bl_r.get('recovery_factor', 0),    new_r.get('recovery_factor', 0)],
        }

        orig_n_st = int(_CACHE['states'].max()) + 1
        bl_lab  = (f'Baseline\nn_st={orig_n_st} depth={params.get("max_depth","?")}')
        new_lab = (f'Modified\nn_st={n_st_new} depth={depth5_w.value} '
                   f'lr={lr5_w.value:.3f}')

        fig, axes = plt.subplots(1, len(metrics_cmp), figsize=(16, 4))
        for ax, (name, (bv, nv)) in zip(axes, metrics_cmp.items()):
            ax.bar([bl_lab, new_lab], [bv, nv], color=['#3498DB', '#E67E22'],
                   width=0.5, edgecolor='white')
            for xi, v in enumerate([bv, nv]):
                ax.text(xi, v * 1.02 + 0.001, f'{v:.2f}',
                        ha='center', fontsize=9, fontweight='bold')
            ax.set_title(name, fontsize=10)
            ymax = max(bv, nv) * 1.35 + 0.01
            ax.set_ylim(0, ymax if ymax > 0 else 1)
        plt.suptitle('Baseline vs Modified Configuration', fontsize=12)
        plt.tight_layout()
        plt.show()

        print(f'\n  {"Metric":<22} {"Baseline":>12} {"Modified":>12} {"Delta":>10}')
        print('  ' + '─' * 58)
        for name, (bv, nv) in metrics_cmp.items():
            d = nv - bv
            print(f'  {name:<22} {bv:>12.3f} {nv:>12.3f} {"+" if d>=0 else ""}{d:>9.3f}')


def _run_wfo5_comparison(_):
    with out5_wfo:
        clear_output(wait=True)
        if 'df_aligned' not in _CACHE:
            print('Run Section 1 first.'); return
        df_al   = _CACHE['df_aligned']
        params  = _CACHE['params']
        tf      = _CACHE['tf']
        broker  = _CACHE['broker']
        balance = _CACHE['balance']
        if not params:
            print('No Optuna params — run optimize first.'); return

        wfo_mode = _CACHE.get('wfo_mode', 'standard')
        cfg_map  = WFO_PARAMS_FAST if wfo_mode == 'fast' else WFO_PARAMS
        cfg      = cfg_map.get(tf.upper(), WFO_PARAMS['H1'])
        n_states = int(params.get('n_states', 3))
        base_kws = {k: params[k] for k in ['max_depth','learning_rate','n_estimators',
                    'subsample','min_child_weight','gamma','reg_alpha',
                    'colsample_bytree','scale_pos_weight'] if k in params}

        mod_kws = dict(base_kws)
        mod_kws['max_depth']        = depth5_w.value
        mod_kws['learning_rate']    = lr5_w.value
        mod_kws['reg_alpha']        = alpha5_w.value
        mod_kws['reg_lambda']       = reg_lambda5_w.value
        mod_kws['min_child_weight'] = min_child_weight5_w.value

        print('Running WFO with baseline params ...')
        wfo_base = _run_wfo(df_al, n_states=n_states, tf=tf, balance=balance, broker=broker,
                            xgb_kwargs=base_kws, **cfg, wfo_mode=wfo_mode)
        print('Running WFO with modified params ...')
        wfo_mod  = _run_wfo(df_al, n_states=n_states, tf=tf, balance=balance, broker=broker,
                            xgb_kwargs=mod_kws, **cfg, wfo_mode=wfo_mode)

        rows = [
            ('wfo_score',       wfo_base['wfo_score'],       wfo_mod['wfo_score']),
            ('n_valid_windows', wfo_base['n_valid_windows'],  wfo_mod['n_valid_windows']),
            ('median_trades',   wfo_base['median_trades'],    wfo_mod['median_trades']),
            ('std_sharpe',      wfo_base['std_sharpe'],       wfo_mod['std_sharpe']),
            ('wfe_ratio',       wfo_base['wfe_ratio'],        wfo_mod['wfe_ratio']),
        ]
        print(f'\n  {"Metric":<22} {"Baseline":>12} {"Modified":>12} {"Delta":>10}')
        print('  ' + '─' * 60)
        for name, bv, mv in rows:
            d = mv - bv
            print(f'  {name:<22} {bv:>12.3f} {mv:>12.3f} {"+" if d>=0 else ""}{d:>9.3f}')

        bs = wfo_base['window_scores']
        ms = wfo_mod['window_scores']
        n  = max(len(bs), len(ms))
        xs = range(n)
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar([x - 0.2 for x in xs[:len(bs)]], bs, width=0.4,
               color='#3498DB', label=f'Baseline (WFO={wfo_base["wfo_score"]:.3f})', alpha=0.8)
        ax.bar([x + 0.2 for x in xs[:len(ms)]], ms, width=0.4,
               color='#E67E22', label=f'Modified (WFO={wfo_mod["wfo_score"]:.3f})', alpha=0.8)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_xlabel('Window index')
        ax.set_ylabel('OOS score')
        ax.set_title(f'WFO Window Scores: Baseline vs Modified [{tf}]')
        ax.legend()
        plt.tight_layout()
        plt.show()


run_btn5.on_click(_run_sensitivity)
run_wfo5_btn.on_click(_run_wfo5_comparison)
display(widgets.VBox([
    n_states5_w, depth5_w, lr5_w, alpha5_w,
    reg_lambda5_w, min_child_weight5_w,
    widgets.HBox([run_btn5, run_wfo5_btn]),
    out5,
    out5_wfo,
]))

---
## Section 6 — Cross-Validation Path Inspector
Runs CPCV (C(6,2)=15 paths) or the inner WFO IS cross-validation on the loaded data
to visualise per-path score distribution. Helps diagnose whether a config is driven by
one lucky path or is consistently profitable across all market regimes.

In [ ]:
cv_method_w = widgets.RadioButtons(
    options=['WFO IS CV', 'CPCV'],
    value='WFO IS CV',
    description='CV Method:',
    style={'description_width': 'initial'},
)
run_cv_btn  = widgets.Button(description='Run CV Inspector', button_style='danger',
                             layout=widgets.Layout(width='180px'))
out6        = widgets.Output()


def _run_cv_inspector(_):
    with out6:
        clear_output(wait=True)
        if 'df_aligned' not in _CACHE:
            print('Run Section 1 first.'); return

        df_al   = _CACHE['df_aligned']
        params  = _CACHE['params']
        tf      = _CACHE['tf']
        broker  = _CACHE['broker']
        balance = _CACHE['balance']

        if not params:
            print('No Optuna params — run optimize first.'); return

        n_states = int(params.get('n_states', 3))
        xgb_kws  = {k: params[k] for k in ['max_depth','learning_rate','n_estimators',
                    'subsample','min_child_weight','gamma','reg_alpha',
                    'colsample_bytree','scale_pos_weight'] if k in params}

        method = cv_method_w.value
        print(f'Running {method} on {tf} data ({len(df_al):,} bars) ...')

        if method == 'CPCV':
            cpcv = compute_cpcv_score(
                df_al, balance=balance, broker=broker, tf=tf,
                n_states=n_states, xgb_kwargs=xgb_kws,
            )
            if isinstance(cpcv, dict):
                path_scores = cpcv.get('path_scores', [])
            else:
                print(f'CPCV returned score={cpcv:.4f} (no per-path breakdown available)'); return
        else:  # WFO IS CV
            wfo_mode = _CACHE.get('wfo_mode', 'standard')
            cfg_map  = WFO_PARAMS_FAST if wfo_mode == 'fast' else WFO_PARAMS
            cfg      = cfg_map.get(tf.upper(), WFO_PARAMS['H1'])
            wfo = _run_wfo(df_al, n_states=n_states, tf=tf, balance=balance,
                           broker=broker, xgb_kwargs=xgb_kws, **cfg, wfo_mode=wfo_mode)
            path_scores = wfo.get('window_scores', [])

        if not path_scores:
            print('No scores returned — check data and params.'); return

        valid_scores = [s for s in path_scores if s is not None and s > -50.0]
        n_pos = sum(1 for s in valid_scores if s > 0)
        consistency_pct = (n_pos / len(valid_scores) * 100) if valid_scores else 0.0

        # ── Boxplot of per-path scores ──────────────────────────────────────
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.boxplot([s for s in path_scores if s is not None and s > -50.0],
                    vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#3498DB', alpha=0.5))
        ax1.axhline(0, color='red', lw=0.8, ls='--')
        ax1.axhline(0.5, color='green', lw=0.8, ls=':', label='score=0.5 target')
        ax1.set_title(f'{method} Score Distribution')
        ax1.set_ylabel('Score')
        ax1.legend(fontsize=8)

        colors_bar = ['#27AE60' if (s is not None and s > 0) else '#E74C3C'
                      for s in path_scores]
        bar_vals   = [s if (s is not None and s > -50.0) else 0.0 for s in path_scores]
        ax2.bar(range(len(bar_vals)), bar_vals, color=colors_bar, edgecolor='white', lw=0.5)
        ax2.axhline(0, color='black', lw=0.8)
        ax2.set_xlabel('Path / Window index')
        ax2.set_ylabel('Score')
        ax2.set_title(f'{method} Per-Path Scores')

        plt.suptitle(
            f'{method} Inspector — {tf}  |  '
            f'Consistency: {consistency_pct:.0f}% ({n_pos}/{len(valid_scores)} paths > 0)',
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

        print(f'\nConsistency score: {consistency_pct:.1f}% of valid paths are profitable')
        print(f'Median score:      {float(np.median(valid_scores)):.4f}')
        print(f'Std score:         {float(np.std(valid_scores)):.4f}')
        print(f'Min / Max:         {min(valid_scores):.4f} / {max(valid_scores):.4f}')


run_cv_btn.on_click(_run_cv_inspector)
display(widgets.VBox([cv_method_w, run_cv_btn, out6]))
